In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1] / "src"))

# Gross Code — LER vs Physical Error Rate (Importance Sampling)

Uses weight-stratified importance sampling (`importance_sampling.py`, based on arXiv:2511.15177) to estimate the logical error rate of the [[144,12,12]] gross code under three scenarios:

1. **Memory experiment** — bare BB syndrome rounds via `BBCodeSimulator(BB_144_12_12)`
2. **LPU X̄₁ circuit** — full gauging-measurement circuit for the X̄₁ logical operator
3. **LPU Z̄₁ circuit** — full gauging-measurement circuit for the Z̄₁ logical operator

**Decoder:** RelayBP (Rust-native) for all three.

### Why IS for the gross code?
Direct MC needs thousands of shots per p value to see any logical errors below threshold. The gross-code LPU circuits in particular run at ~3× the per-shot cost of the bare memory circuit, so a direct sweep over a 2-decade $p$ range is expensive. Importance sampling does one fixed-budget pass per circuit and gives the entire LER vs $p$ curve from a single reweighting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from bb_code_sim import BBCodeSimulator, BB_144_12_12, RelayBPDecoder
from surface_code_sim import ErrorModel
import gross_code_lpu_tdg as tdg
from importance_sampling import importance_sample

SEED = 42

# Common IS parameters
P_REF             = 0.005
WEIGHTS           = list(range(1, 11))    # sample at fault weights 1..10
SHOTS_PER_WEIGHT  = 150                   # samples per weight per circuit
P_TARGETS         = np.logspace(-3.5, -1.5, 30)  # reweight to ~3.2e-4 .. 3.2e-2

# LPU parameters
C, D_INIT = 10, 12

print(f'Gross code: [[{BB_144_12_12.l*BB_144_12_12.m*2}, 12, {BB_144_12_12.distance}]]')
print(f'IS parameters: p_ref={P_REF}, weights={WEIGHTS}, shots_per_weight={SHOTS_PER_WEIGHT}')
print(f'Reweighting to {len(P_TARGETS)} target p values from {P_TARGETS[0]:.4f} to {P_TARGETS[-1]:.4f}')

## 1. Memory experiment — IS sweep

Build one Z-memory circuit at `p_ref`, run IS, reweight to the full $p$ grid.

In [ ]:
em_ref = ErrorModel.symmetric(P_REF)

sim_mem = BBCodeSimulator(BB_144_12_12)
mem_circuit = sim_mem.build_circuit(em_ref, rounds=BB_144_12_12.distance)
print(f'Memory circuit: {mem_circuit.num_qubits} qubits, {mem_circuit.num_detectors} detectors')

t0 = time.perf_counter()
mem_is = importance_sample(
    mem_circuit, RelayBPDecoder(),
    p_ref=P_REF, p_values=P_TARGETS,
    weights=WEIGHTS, shots_per_weight=SHOTS_PER_WEIGHT,
    seed=SEED,
)
print(f'Memory IS done in {time.perf_counter()-t0:.1f}s')
print(f'  N_expanded={mem_is.spectrum.n_expanded}, q_base={mem_is.spectrum.q_base:.5f}')
print()
print('  Failure spectrum:')
for w, T, F in zip(mem_is.spectrum.weights, mem_is.spectrum.trials, mem_is.spectrum.failures):
    print(f'    w={w:>2}: F/T = {F:>4}/{T}  =  {F/T:.3f}')

## 2. LPU X̄₁ circuit — IS sweep

In [ ]:
x1_circuit = tdg.build_logical_x1_circuit(em_ref, C=C, d_init=D_INIT)
print(f'X̄₁ LPU circuit: {x1_circuit.num_qubits} qubits, {x1_circuit.num_detectors} detectors')

t0 = time.perf_counter()
x1_is = importance_sample(
    x1_circuit, RelayBPDecoder(),
    p_ref=P_REF, p_values=P_TARGETS,
    weights=WEIGHTS, shots_per_weight=SHOTS_PER_WEIGHT,
    seed=SEED,
)
print(f'X̄₁ IS done in {time.perf_counter()-t0:.1f}s')
print(f'  N_expanded={x1_is.spectrum.n_expanded}, q_base={x1_is.spectrum.q_base:.5f}')
print()
print('  Failure spectrum:')
for w, T, F in zip(x1_is.spectrum.weights, x1_is.spectrum.trials, x1_is.spectrum.failures):
    print(f'    w={w:>2}: F/T = {F:>4}/{T}  =  {F/T:.3f}')

## 3. LPU Z̄₁ circuit — IS sweep

In [ ]:
z1_circuit = tdg.build_logical_z1_circuit(em_ref, C=C, d_init=D_INIT)
print(f'Z̄₁ LPU circuit: {z1_circuit.num_qubits} qubits, {z1_circuit.num_detectors} detectors')

t0 = time.perf_counter()
z1_is = importance_sample(
    z1_circuit, RelayBPDecoder(),
    p_ref=P_REF, p_values=P_TARGETS,
    weights=WEIGHTS, shots_per_weight=SHOTS_PER_WEIGHT,
    seed=SEED,
)
print(f'Z̄₁ IS done in {time.perf_counter()-t0:.1f}s')
print(f'  N_expanded={z1_is.spectrum.n_expanded}, q_base={z1_is.spectrum.q_base:.5f}')
print()
print('  Failure spectrum:')
for w, T, F in zip(z1_is.spectrum.weights, z1_is.spectrum.trials, z1_is.spectrum.failures):
    print(f'    w={w:>2}: F/T = {F:>4}/{T}  =  {F/T:.3f}')

## 4. Comparison plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for result, label, color in [
    (mem_is, 'Memory (d=12 rounds)',          'steelblue'),
    (x1_is,  f'LPU X̄₁ (C={C}, d_init={D_INIT})', 'tomato'),
    (z1_is,  f'LPU Z̄₁ (C={C}, d_init={D_INIT})', 'seagreen'),
]:
    lo = np.maximum(result.P_logical - result.P_logical_se, 1e-15)
    hi = result.P_logical + result.P_logical_se
    ax.fill_between(result.p_values, lo, hi, color=color, alpha=0.25)
    ax.plot(result.p_values, result.P_logical, '-', color=color, label=label, lw=2)

ax.axhline(0.5, color='grey', ls='--', lw=0.8, label='50% (random)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical error rate $p$')
ax.set_ylabel('Logical error rate (importance-sampled)')
ax.set_title(f'[[144,12,12]] Gross Code — LER vs $p$ via IS  (RelayBP decoder)\n'
             f'{len(WEIGHTS)} weights × {SHOTS_PER_WEIGHT} shots/circuit, reweighted to {len(P_TARGETS)} p points')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. LER ratios — LPU overhead at fixed $p$

The LPU circuits do more work than the bare memory circuit (extra rounds, edge qubits). At any given $p$, how much LER does that cost?

In [ ]:
ratio_x1 = x1_is.P_logical / np.maximum(mem_is.P_logical, 1e-15)
ratio_z1 = z1_is.P_logical / np.maximum(mem_is.P_logical, 1e-15)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(P_TARGETS, ratio_x1, 'o-', color='tomato',   label='X̄₁ / memory', lw=2)
ax.plot(P_TARGETS, ratio_z1, 's-', color='seagreen', label='Z̄₁ / memory', lw=2)
ax.axhline(1.0, color='grey', ls='--', lw=0.8)
ax.set_xscale('log')
ax.set_xlabel('Physical error rate $p$')
ax.set_ylabel('LPU LER / memory LER')
ax.set_title('LPU overhead relative to bare memory experiment')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()